# 06 - Simulation and Optimization

This optional notebook covers the fast-moving improvement loop:

```text
baseline -> simulated or live sessions -> evaluation
         -> failure analysis -> recommendation
         -> controlled validation -> promotion
```

You will:

1. Define a simulated multi-turn user.
2. Connect insights to recommendation jobs.
3. Understand configuration bundles and Gateway-backed A/B tests.
4. Clean up the module's AWS resources.

**Estimated time:** 35-60 minutes  
**Feature status:** Simulated dataset evaluation and AgentCore Insights are public preview as of August 14, 2026. Recommendations, config bundles, and A/B testing are fast-moving optional extensions. Verify your installed CLI help and regional support before running them.

## 1. Verify the current surface

```bash
agentcore run insights --help
agentcore run recommendation --help
agentcore add config-bundle --help
agentcore run ab-test --help
```

This notebook uses the AgentCore Python SDK `1.21.0` and AgentCore CLI `0.27.0` surfaces verified on August 14, 2026.

## 2. Define a simulated actor

A useful simulated actor has:

- a concrete goal
- relevant context
- behavioral traits that affect the conversation
- a maximum turn count
- assertions that define success

Do not use simulation to manufacture a large score. Use it to reach interaction states that are rare, expensive, or awkward to script manually.

In [ ]:
from bedrock_agentcore.evaluation import (
    ActorProfile,
    Dataset,
    SimulatedScenario,
)

simulated_dataset = Dataset(
    scenarios=[
        SimulatedScenario(
            scenario_id="ambiguous-portland-follow-up",
            scenario_description=(
                "A beginner asks about Portland without a state, "
                "then clarifies and requests a Seattle comparison."
            ),
            actor_profile=ActorProfile(
                traits={
                    "technical_level": "beginner",
                    "communication_style": "brief follow-up questions",
                },
                context=(
                    "The user means Portland, Oregon and is comparing "
                    "potential relocation cities."
                ),
                goal=(
                    "Obtain the workshop facts for Portland, OR and "
                    "a correct comparison with Seattle, WA."
                ),
            ),
            input="Tell me about Portland.",
            max_turns=6,
            assertions=[
                "The agent should clarify the state before looking up Portland.",
                "The final comparison should use Portland, OR and Seattle, WA.",
                "The agent should not invent facts for Portland, ME.",
            ],
        )
    ]
)
simulated_dataset.model_dump()

## 3. Configure the simulation runner

The actor model plays the user. The deployed `CityAnalyst` remains the system under test. These are separate roles and may use different models.

In [ ]:
import os

from bedrock_agentcore.evaluation import (
    CloudWatchAgentSpanCollector,
    EvaluationRunConfig,
    EvaluatorConfig,
    OnDemandEvaluationDatasetRunner,
    SimulationConfig,
)
from src.workshop_utils import RuntimeInvoker, load_runtime_info

runtime = load_runtime_info()
invoker = RuntimeInvoker(runtime)
collector = CloudWatchAgentSpanCollector(
    log_group_name=runtime.log_group_name,
    region=runtime.region,
    max_wait_seconds=180,
    poll_interval_seconds=10,
)

actor_model = os.getenv(
    "AGENTCORE_SIMULATOR_MODEL_ID",
    "global.anthropic.claude-sonnet-4-6",
)
simulation_run_config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(
        evaluator_ids=[
            "Builtin.GoalSuccessRate",
            "Builtin.InstructionFollowing",
            "Builtin.Helpfulness",
        ]
    ),
    evaluation_delay_seconds=0,
    max_concurrent_scenarios=1,
    simulation_config=SimulationConfig(model_id=actor_model),
)

In [ ]:
RUN_SIMULATION = False

if RUN_SIMULATION:
    simulation_result = OnDemandEvaluationDatasetRunner(
        region=runtime.region
    ).run(
        config=simulation_run_config,
        dataset=simulated_dataset,
        agent_invoker=invoker,
        span_collector=collector,
    )
    print(simulation_result.model_dump_json(indent=2))
else:
    print(
        "Simulation invokes both an actor model and the agent. "
        "Set RUN_SIMULATION=True when you are ready."
    )

## 4. Turn low scores into failure hypotheses

Before changing the prompt, inspect:

- the user goal and turn where failure began
- tool selection and parameters
- tool result quality
- instruction-following errors
- whether the evaluator itself misunderstood the case

A prompt change is only one possible intervention. The root cause may be a tool description, missing validation, session-state bug, or inadequate ground truth.

## 5. Insights jobs

Insights can cluster failure patterns across recent sessions. Current CLI insight IDs include:

- `Builtin.Insight.FailureAnalysis`
- `Builtin.Insight.UserIntent`
- `Builtin.Insight.ExecutionSummary`

Example:

```bash
agentcore run insights \
  --runtime CityAnalyst \
  --insights Builtin.Insight.FailureAnalysis Builtin.Insight.UserIntent \
  --evaluator Builtin.GoalSuccessRate \
  --lookback-days 7 \
  --name CityAnalystFailures \
  --wait

agentcore view insights
```

Treat clusters as hypotheses. Validate them against individual traces and human review before changing the system.

## 6. Recommendation jobs

Recommendation can optimize a system prompt or tool descriptions using traces and an evaluator as the signal.

```bash
agentcore run recommendation \
  --type system-prompt \
  --runtime CityAnalyst \
  --evaluator Builtin.GoalSuccessRate \
  --prompt-file app/CityAnalyst/system_prompt.txt \
  --lookback 7 \
  --run CityPromptRecommendation \
  --wait

agentcore view recommendation
```

A recommendation is a candidate, not an automatic truth. Save the baseline, review the proposed change, rerun the curated dataset, and compare failure-level results before promotion.

## 7. Configuration bundles

Configuration bundles version runtime configuration separately from application code. They are useful for prompt variants and controlled experiments.

Example component file:

In [ ]:
import json
from pathlib import Path

control_components = {
    "{{runtime:CityAnalyst}}": {
        "configuration": {
            "systemPrompt": Path(
                "app/CityAnalyst/system_prompt.txt"
            ).read_text()
        }
    }
}
Path("generated").mkdir(exist_ok=True)
Path("generated/control_bundle.json").write_text(
    json.dumps(control_components, indent=2) + "\n"
)
control_components

Add and deploy a bundle:

```bash
agentcore add config-bundle \
  --name CityPromptControl \
  --description "Baseline CityAnalyst prompt" \
  --components-file generated/control_bundle.json \
  --branch main \
  --commit-message "Workshop baseline"

agentcore deploy
```

Use `agentcore config-bundle versions` and `agentcore config-bundle diff` to make the comparison auditable.

## 8. Gateway-backed A/B tests

AgentCore A/B testing splits traffic at a deployed Gateway. It is not a shortcut for a notebook-only comparison.

Prerequisites:

- a deployed Gateway
- two config-bundle versions or two Gateway targets
- online evaluation configured for the compared traffic
- enough representative traffic to make the result meaningful

Example:

```bash
agentcore run ab-test \
  --name CityPromptExperiment \
  --gateway <GATEWAY_NAME> \
  --runtime CityAnalyst \
  --control-bundle CityPromptControl \
  --control-version LATEST \
  --treatment-bundle CityPromptTreatment \
  --treatment-version LATEST \
  --online-eval CityQualityMonitor \
  --control-weight 50 \
  --treatment-weight 50
```

Promote only after reviewing effect size, evaluator reliability, segment behavior, operational metrics, and rollback readiness.

## 9. The complete improvement loop

| Stage | Evidence |
|---|---|
| Baseline | Versioned agent, dataset, evaluator, and score distribution |
| Failure analysis | Traces, explanations, human review, and optional insights |
| Change | Prompt, tool description, validation, model, or code |
| Offline validation | Same curated dataset and evaluator versions |
| Controlled rollout | Online sampling or Gateway A/B test |
| Promotion | Documented decision and rollback point |

Never optimize only the aggregate score. Inspect the failure categories the change improves and the categories it may regress.

## 10. Cleanup

The cleanup helper temporarily deploys an empty project specification, allowing CloudFormation to remove resources managed by this module, and then restores the checked-in configuration.

It does not delete external Lambda functions, customer-managed KMS keys, or log groups retained by policy.

In [ ]:
import subprocess

RUN_CLEANUP = False

if RUN_CLEANUP:
    subprocess.run(
        ["python", "src/cleanup.py", "--yes"],
        check=True,
    )
else:
    print(
        "Set RUN_CLEANUP=True after completing the workshop, "
        "then verify retained resources in the AWS console."
    )

## Module complete

You can now follow AgentCore evaluation as one coherent progression:

1. run one agent
2. observe sessions, traces, and tool spans
3. evaluate recent behavior with built-ins
4. add stable ground truth and datasets
5. create focused, calibrated custom evaluators
6. operate batch and online monitoring
7. simulate harder users and validate improvements

The core habit is simple: define the failure, collect the right evidence, use the narrowest evaluator that can detect it, and validate the evaluator before automating decisions.